# Análise Qualitativa e Macro-Grupo: Classificador × Verificador

**Objetivo:** Avaliar o classificador por **macro-grupo** e gerar **exemplos visuais** de acertos e erros por tipo de par.

---


In [ ]:
import os, json, random, time, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
import timm

from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, f1_score, precision_score, recall_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_DATA = '/content/local_data'
os.makedirs(LOCAL_DATA, exist_ok=True)

DATASET_DIR = os.path.join(LOCAL_DATA, 'dogs')
ARCHIVE_PATH = os.path.join(DRIVE_BASE, 'dogs', 'dog.tar.gz')

if not os.path.isdir(DATASET_DIR):
    assert os.path.isfile(ARCHIVE_PATH), f"Arquivo nao encontrado: {ARCHIVE_PATH}"
    os.makedirs(DATASET_DIR, exist_ok=True)
    print("Extraindo dataset para SSD local...")
    t0 = time.time()
    subprocess.run(f'tar -xzf "{ARCHIVE_PATH}" -C "{DATASET_DIR}"', shell=True, check=True)
    print(f"Extracao concluida em {time.time()-t0:.1f}s")
else:
    print("Dataset ja existe localmente.")

VERIFIER_WEIGHTS_DRIVE = os.path.join(DRIVE_BASE, 'dogs', 'best_arcface.pth')
VERIFIER_WEIGHTS = os.path.join(LOCAL_DATA, 'best_arcface.pth')
if not os.path.isfile(VERIFIER_WEIGHTS):
    subprocess.run(f'cp "{VERIFIER_WEIGHTS_DRIVE}" "{VERIFIER_WEIGHTS}"', shell=True, check=True)

CLASSIFIER_WEIGHTS = f"{DRIVE_BASE}/classification_exp/results/best_model_stanford.pth"
LABEL_ENCODER_PATH = f"{DRIVE_BASE}/classification_exp/results/label_encoder_stanford.json"
MACRO_GROUPS_PATH  = f"{DRIVE_BASE}/classification_exp/results/breed_macro_groups.json"
VERIF_CSV          = f"{DRIVE_BASE}/dogs/split/verification_test_pairs.csv"

RESULTS_DIR = f"{DRIVE_BASE}/classification_exp/results/qualitative_analysis"
os.makedirs(RESULTS_DIR, exist_ok=True)

df_verif = pd.read_csv(VERIF_CSV)
print(f"Pares de verificacao: {len(df_verif)}")
print(df_verif['pair_type'].value_counts())


## Definição dos Modelos


In [ ]:
class EmbeddingModel(nn.Module):
    def __init__(self, embedding_dim=512, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model('convnext_small.fb_in22k_ft_in1k', pretrained=pretrained, num_classes=0)
        self.embedding = nn.Sequential(
            nn.Linear(self.backbone.num_features, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
        )
    def forward(self, x):
        features = self.backbone(x)
        embeddings = self.embedding(features)
        return F.normalize(embeddings, p=2, dim=1)

def create_classifier(num_classes):
    model = models.convnext_tiny(weights=None)
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, num_classes)
    return model


## Carregar Modelos e Macro-Grupos


In [ ]:
with open(LABEL_ENCODER_PATH, 'r') as f:
    label_encoder = json.load(f)
num_classes = label_encoder['num_classes']
idx_to_label = {int(k): v for k, v in label_encoder['idx_to_label'].items()}

with open(MACRO_GROUPS_PATH, 'r') as f:
    macro_data = json.load(f)
breed_to_group = macro_data['breed_to_group']

group_to_breeds = {}
for breed, group in breed_to_group.items():
    group_to_breeds.setdefault(group, []).append(breed)
print(f"Total de macro-grupos: {len(group_to_breeds)}")
for g, breeds in sorted(group_to_breeds.items()):
    print(f"  {g}: {', '.join(sorted(breeds))}")

classifier = create_classifier(num_classes)
clf_state = torch.load(CLASSIFIER_WEIGHTS, map_location=DEVICE, weights_only=False)
clf_state = clf_state.get('model_state_dict', clf_state)
clf_state = {k.replace('_orig_mod.', '').replace('module.', ''): v for k, v in clf_state.items()}
classifier.load_state_dict(clf_state)
classifier.to(DEVICE).eval()
print(f"Classificador carregado ({num_classes} classes).")

verifier = EmbeddingModel(embedding_dim=512, pretrained=False)
verif_ckpt = torch.load(VERIFIER_WEIGHTS, map_location=DEVICE, weights_only=False)
verifier.load_state_dict(verif_ckpt['backbone'])
verifier.to(DEVICE).eval()
print("Verificador carregado.")


## Dataset e Inferência Dupla


In [ ]:
eval_transform = T.Compose([
    T.Resize((224, 224), interpolation=T.InterpolationMode.BILINEAR),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class PairDataset(Dataset):
    def __init__(self, df, root_dir, transform):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p1 = os.path.join(self.root_dir, row['filename1'])
        p2 = os.path.join(self.root_dir, row['filename2'])
        img1 = Image.open(p1).convert('RGB')
        img2 = Image.open(p2).convert('RGB')
        t1 = self.transform(img1)
        t2 = self.transform(img2)
        return t1, t2, int(row['label']), row.get('pair_type', 'unknown'), p1, p2

test_loader = DataLoader(PairDataset(df_verif, DATASET_DIR, eval_transform),
                         batch_size=64, shuffle=False, num_workers=2)


In [ ]:
all_labels = []
all_pair_types = []
all_paths_1 = []
all_paths_2 = []
sims_verifier = []
breed_overlaps = []
top1_match = []
top1_group_match = []
clf_probs_1 = []
clf_probs_2 = []
all_top1_breed_1 = []
all_top1_breed_2 = []
all_top1_group_1 = []
all_top1_group_2 = []
topk_match = {3: [], 5: []}

print("Executando inferencia dupla...")
with torch.no_grad():
    for img1, img2, lbl, ptype, p1, p2 in tqdm(test_loader):
        img1, img2 = img1.to(DEVICE), img2.to(DEVICE)
        all_labels.extend(lbl.numpy())
        all_pair_types.extend(ptype)
        all_paths_1.extend(p1)
        all_paths_2.extend(p2)

        with torch.amp.autocast('cuda'):
            emb1 = verifier(img1)
            emb2 = verifier(img2)
            cos_sim = F.cosine_similarity(emb1, emb2, dim=1)
            norm_sim = (cos_sim + 1.0) / 2.0
            sims_verifier.extend(norm_sim.cpu().numpy())

            probs1 = torch.softmax(classifier(img1), dim=1)
            probs2 = torch.softmax(classifier(img2), dim=1)
            overlap = torch.sum(probs1 * probs2, dim=1)
            breed_overlaps.extend(overlap.cpu().numpy())

            top1_1 = probs1.argmax(dim=1)
            top1_2 = probs2.argmax(dim=1)
            top1_match.extend((top1_1 == top1_2).cpu().numpy())

            for k in [3, 5]:
                topk1 = probs1.topk(k, dim=1).indices
                topk2 = probs2.topk(k, dim=1).indices
                for i in range(len(lbl)):
                    set1 = set(topk1[i].cpu().numpy().tolist())
                    set2 = set(topk2[i].cpu().numpy().tolist())
                    topk_match[k].append(len(set1 & set2) > 0)

            for i in range(len(lbl)):
                b1 = idx_to_label.get(top1_1[i].item(), 'Unknown')
                b2 = idx_to_label.get(top1_2[i].item(), 'Unknown')
                g1 = breed_to_group.get(b1, 'Unknown')
                g2 = breed_to_group.get(b2, 'Unknown')
                top1_group_match.append(g1 == g2)
                all_top1_breed_1.append(b1)
                all_top1_breed_2.append(b2)
                all_top1_group_1.append(g1)
                all_top1_group_2.append(g2)

            clf_probs_1.append(probs1.cpu().numpy())
            clf_probs_2.append(probs2.cpu().numpy())

all_labels = np.array(all_labels)
sims_verifier = np.array(sims_verifier)
breed_overlaps = np.array(breed_overlaps)
top1_match = np.array(top1_match)
top1_group_match = np.array(top1_group_match)
all_pair_types = np.array(all_pair_types)
clf_probs_1 = np.concatenate(clf_probs_1, axis=0)
clf_probs_2 = np.concatenate(clf_probs_2, axis=0)
for k in topk_match:
    topk_match[k] = np.array(topk_match[k])

print(f"Inferencia concluida! {len(all_labels)} pares.")


## Funções de Avaliação


In [ ]:
def evaluate(scores, labels, pair_types, name="Strategy"):
    auc = roc_auc_score(labels, scores)
    fpr, tpr, thresholds = roc_curve(labels, scores)
    best_idx = np.argmax(tpr - fpr)
    best_thresh = thresholds[best_idx]
    preds = (scores >= best_thresh).astype(int)
    acc = accuracy_score(labels, preds)
    f1v = f1_score(labels, preds)
    prec = precision_score(labels, preds)
    rec = recall_score(labels, preds)

    mask_hard = (pair_types == 'negative_same_breed')
    acc_hard = accuracy_score(labels[mask_hard], preds[mask_hard]) if mask_hard.sum() > 0 else 0
    mask_easy = (pair_types == 'negative_random')
    acc_easy = accuracy_score(labels[mask_easy], preds[mask_easy]) if mask_easy.sum() > 0 else 0
    mask_pos = (pair_types == 'positive')
    acc_pos = accuracy_score(labels[mask_pos], preds[mask_pos]) if mask_pos.sum() > 0 else 0

    print(f"--- {name} ---")
    print(f"AUC: {auc:.4f} | Acc: {acc:.4f} | F1: {f1v:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f}")
    print(f"Thresh: {best_thresh:.4f} | Acc Pos: {acc_pos:.4f} | Acc Hard: {acc_hard:.4f} | Acc Easy: {acc_easy:.4f}")
    print()
    return {'name': name, 'auc': auc, 'acc': acc, 'f1': f1v, 'prec': prec, 'rec': rec,
            'thresh': best_thresh, 'preds': preds,
            'acc_hard': acc_hard, 'acc_easy': acc_easy, 'acc_pos': acc_pos}

def evaluate_binary(preds, labels, pair_types, name="Strategy"):
    acc = accuracy_score(labels, preds)
    f1v = f1_score(labels, preds)
    prec = precision_score(labels, preds)
    rec = recall_score(labels, preds)

    mask_hard = (pair_types == 'negative_same_breed')
    acc_hard = accuracy_score(labels[mask_hard], preds[mask_hard]) if mask_hard.sum() > 0 else 0
    mask_easy = (pair_types == 'negative_random')
    acc_easy = accuracy_score(labels[mask_easy], preds[mask_easy]) if mask_easy.sum() > 0 else 0
    mask_pos = (pair_types == 'positive')
    acc_pos = accuracy_score(labels[mask_pos], preds[mask_pos]) if mask_pos.sum() > 0 else 0

    print(f"--- {name} ---")
    print(f"Acc: {acc:.4f} | F1: {f1v:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f}")
    print(f"Acc Pos: {acc_pos:.4f} | Acc Hard: {acc_hard:.4f} | Acc Easy: {acc_easy:.4f}")
    print()
    return {'name': name, 'acc': acc, 'f1': f1v, 'prec': prec, 'rec': rec,
            'acc_hard': acc_hard, 'acc_easy': acc_easy, 'acc_pos': acc_pos, 'preds': preds}

print("OK")


## Avaliação Quantitativa Completa


In [ ]:
# 1. Verificador Solo
res_verif = evaluate(sims_verifier, all_labels, all_pair_types, "Verificador Solo (ArcFace)")
verif_thresh = res_verif['thresh']

# 2. Classificador Solo - Top-1 raca
res_clf_top1 = evaluate_binary(top1_match.astype(int), all_labels, all_pair_types, "Clf Solo (Top-1 Raca)")

# 3. Classificador Solo - Top-1 macro-grupo
res_clf_group = evaluate_binary(top1_group_match.astype(int), all_labels, all_pair_types, "Clf Solo (Top-1 Macro-Grupo)")

# 4. Classificador Solo - Breed Overlap
res_clf_overlap = evaluate(breed_overlaps, all_labels, all_pair_types, "Clf Solo (Breed Overlap)")

# 5. Top-k concordancia
for k in [3, 5]:
    evaluate_binary(topk_match[k].astype(int), all_labels, all_pair_types, f"Clf Solo (Top-{k} Concordancia)")

# 6. Pipeline Bypass (Veto)
def pipeline_bypass(ot, verif_scores, verif_threshold, labels, pair_types, name):
    fmask = (breed_overlaps < ot)
    preds = np.zeros(len(labels), dtype=int)
    preds[~fmask] = (verif_scores[~fmask] >= verif_threshold).astype(int)
    acc = accuracy_score(labels, preds)
    f1v = f1_score(labels, preds)
    mh = (pair_types == 'negative_same_breed')
    me = (pair_types == 'negative_random')
    mp = (pair_types == 'positive')
    print(f"--- {name} --- Filtrados: {fmask.sum()} ({fmask.mean()*100:.1f}%)")
    print(f"Acc: {acc:.4f} | F1: {f1v:.4f}")
    print(f"Acc Pos: {accuracy_score(labels[mp], preds[mp]):.4f} | Acc Hard: {accuracy_score(labels[mh], preds[mh]):.4f} | Acc Easy: {accuracy_score(labels[me], preds[me]):.4f}")
    print()
    return {'name': name, 'acc': acc, 'f1': f1v, 'preds': preds}

for ot in [0.01, 0.05, 0.10, 0.20]:
    pipeline_bypass(ot, sims_verifier, verif_thresh, all_labels, all_pair_types, f"Bypass (overlap<{ot})")


## Tabela Comparativa


In [ ]:
rows = []
for name, r in [("Verificador Solo", res_verif),
                ("Clf Top-1 Raca", res_clf_top1),
                ("Clf Top-1 Grupo", res_clf_group),
                ("Clf Overlap", res_clf_overlap)]:
    rows.append({'Estrategia': name, 'Acc': r['acc'], 'F1': r['f1'],
                 'Acc Pos': r['acc_pos'], 'Acc Hard': r['acc_hard'], 'Acc Easy': r['acc_easy']})
df_tab = pd.DataFrame(rows)
print(df_tab.to_string(index=False, float_format=lambda x: f'{x:.4f}'))


## Análise Qualitativa: Exemplos Visuais

Pares de imagens para cada cenário de acerto/erro, com raça e grupo preditos pelo classificador.


In [ ]:
def show_pair_examples(indices, title, n_show=6, figsize=(20, 4)):
    if len(indices) == 0:
        print(f"Nenhum exemplo para: {title}")
        return
    indices = list(indices)
    random.shuffle(indices)
    n_show = min(n_show, len(indices))
    chosen = indices[:n_show]

    fig, axes = plt.subplots(2, n_show, figsize=figsize)
    if n_show == 1:
        axes = axes.reshape(2, 1)
    fig.suptitle(f"{title} ({len(indices)} pares, mostrando {n_show})", fontsize=14, fontweight='bold')

    for col, idx in enumerate(chosen):
        p1 = all_paths_1[idx]
        p2 = all_paths_2[idx]
        img1_pil = Image.open(p1).convert('RGB').resize((224, 224))
        img2_pil = Image.open(p2).convert('RGB').resize((224, 224))

        sim_v = sims_verifier[idx]
        ov = breed_overlaps[idx]
        b1 = all_top1_breed_1[idx]
        b2 = all_top1_breed_2[idx]
        g1 = all_top1_group_1[idx]
        g2 = all_top1_group_2[idx]
        lbl = all_labels[idx]

        axes[0, col].imshow(img1_pil)
        axes[0, col].set_title(f"Raca: {b1}\nGrupo: {g1}", fontsize=7)
        axes[0, col].axis('off')

        axes[1, col].imshow(img2_pil)
        axes[1, col].set_title(f"Raca: {b2}\nGrupo: {g2}", fontsize=7)
        axes[1, col].axis('off')

        color = '#2196F3' if lbl == 1 else '#F44336'
        label_text = "MESMO CAO" if lbl == 1 else "CAES DIFERENTES"
        fig.text((col + 0.5) / n_show, 0.01,
                 f"sim={sim_v:.3f} | overlap={ov:.3f}\n{label_text}",
                 ha='center', fontsize=8, color=color, fontweight='bold')

    plt.tight_layout(rect=[0, 0.06, 1, 0.93])
    safe_title = title.replace(' ', '_').replace(':', '').replace('(', '').replace(')', '').lower()
    fig_path = os.path.join(RESULTS_DIR, f"{safe_title}.png")
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"Salvo: {fig_path}")
    plt.show()

print("Funcao de visualizacao definida.")


### Exemplos: Verificador


In [ ]:
verif_preds = res_verif['preds']
mask_pos = (all_pair_types == 'positive')
mask_hard = (all_pair_types == 'negative_same_breed')
mask_easy = (all_pair_types == 'negative_random')
verif_correct = (verif_preds == all_labels)
verif_wrong = ~verif_correct

show_pair_examples(np.where(mask_pos & verif_correct)[0],  "Positivos: Verificador ACERTOU", n_show=6)
show_pair_examples(np.where(mask_pos & verif_wrong)[0],    "Positivos: Verificador ERROU (Falso Negativo)", n_show=6)
show_pair_examples(np.where(mask_hard & verif_correct)[0], "Hard Neg (mesma raca): Verificador ACERTOU", n_show=6)
show_pair_examples(np.where(mask_hard & verif_wrong)[0],   "Hard Neg (mesma raca): Verificador ERROU (Falso Positivo)", n_show=6)
show_pair_examples(np.where(mask_easy & verif_correct)[0], "Easy Neg (raca diferente): Verificador ACERTOU", n_show=6)
show_pair_examples(np.where(mask_easy & verif_wrong)[0],   "Easy Neg (raca diferente): Verificador ERROU (Falso Positivo)", n_show=6)


### Exemplos: Classificador como Filtro

Onde o classificador erraria se fosse usado como veto (top-1 ou grupo).


In [ ]:
show_pair_examples(np.where(mask_pos & ~top1_match)[0],
                   "Positivos: Clf prediz RACA DIFERENTE (top-1)", n_show=6)
show_pair_examples(np.where(mask_pos & ~top1_group_match)[0],
                   "Positivos: Clf prediz GRUPO DIFERENTE", n_show=6)
show_pair_examples(np.where(mask_hard & top1_match)[0],
                   "Hard Neg: Clf prediz MESMA RACA (nao filtra)", n_show=6)
show_pair_examples(np.where(mask_hard & ~top1_match)[0],
                   "Hard Neg: Clf prediz RACA DIFERENTE (filtra corretamente)", n_show=6)


## Grafico Comparativo


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

strategies = ['Verif. Solo', 'Clf Top-1\nRaca', 'Clf Top-1\nGrupo', 'Clf\nOverlap']
acc_pos_vals  = [res_verif['acc_pos'], res_clf_top1['acc_pos'], res_clf_group['acc_pos'], res_clf_overlap['acc_pos']]
acc_hard_vals = [res_verif['acc_hard'], res_clf_top1['acc_hard'], res_clf_group['acc_hard'], res_clf_overlap['acc_hard']]
acc_easy_vals = [res_verif['acc_easy'], res_clf_top1['acc_easy'], res_clf_group['acc_easy'], res_clf_overlap['acc_easy']]

x = np.arange(len(strategies))
w = 0.6
colors = ['#2196F3', '#FF5722', '#FF9800', '#FFC107']

for ax, vals, title in [(axes[0], acc_pos_vals, 'Acc Positivos'),
                         (axes[1], acc_hard_vals, 'Acc Hard Negatives'),
                         (axes[2], acc_easy_vals, 'Acc Easy Negatives')]:
    bars = ax.bar(x, [v*100 for v in vals], w, color=colors, edgecolor='white')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h+0.5, f'{h:.1f}%', ha='center', fontsize=9, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(strategies, fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 105)
    ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.suptitle('Acuracia por Tipo de Par', fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
fig_path = os.path.join(RESULTS_DIR, 'comparacao_estrategias_por_tipo.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Salvo: {fig_path}")
plt.show()


In [ ]:
print("\n" + "="*70)
print("RESUMO FINAL")
print("="*70)
print(f"  Verificador Solo:     Acc={res_verif['acc']:.4f} | AUC={res_verif['auc']:.4f}")
print(f"  Clf Top-1 Raca:       Acc={res_clf_top1['acc']:.4f}")
print(f"  Clf Top-1 Grupo:      Acc={res_clf_group['acc']:.4f}")
print(f"  Clf Overlap:          Acc={res_clf_overlap['acc']:.4f} | AUC={res_clf_overlap['auc']:.4f}")
print(f"\nResultados salvos em: {RESULTS_DIR}")
